# k-fold cross-validation

_Machine Learning — more_

**Reuse all your data: take turns holding out each slice to test on.**

You must test a model on data it did not train on — otherwise you only measure memorization.

     But a single train/test split wastes data and gives a noisy score.

---

This notebook is a practice scaffold for the **k-fold cross-validation** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

### Step 1 — Make a regression dataset and define the folds

We generate 200 noisy regression examples with 8 features. Then we set up a **5-fold** splitter: it will carve the data into 5 equal slices, and on each pass one slice is held out for testing while the other four train the model. Shuffling first (with a fixed seed) makes sure the folds are a fair mix rather than depending on row order.

In [ ]:
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold

# 200 examples, 8 features, with a moderate amount of label noise.
X, y = make_regression(
    n_samples=200,
    n_features=8,
    noise=12.0,
    random_state=0,
)

# 5 folds; shuffle so the slices are a fair mix of the rows.
model = KFold(n_splits=5, shuffle=True, random_state=0)

### Step 2 — Score every fold

`cross_val_score` trains a fresh linear model on each fold's training slice and scores it on the held-out slice. We ask for `neg_mean_squared_error` because scikit-learn always *maximises* its score, so it reports error as a negative number; flipping the sign gives us the actual MSE per fold. Five folds means five independent error estimates.

In [ ]:
# Negative MSE per fold (sklearn maximizes, so error is reported negative).
scores = cross_val_score(
    LinearRegression(), X, y,
    cv=model,
    scoring="neg_mean_squared_error",
)

# Flip the sign to get the real mean-squared error for each fold.
mse_per_fold = -scores
for j, m in enumerate(mse_per_fold, 1):
    print("fold %d: MSE = %.3f" % (j, m))

### Step 3 — Summarise across folds

The cross-validation estimate is just the **mean** of the five fold errors, which is far more stable than any single train/test split. The **standard deviation** tells us how much the score wobbles from fold to fold — a small spread means the estimate is trustworthy.

In [ ]:
cv_mse = mse_per_fold.mean()
cv_std = mse_per_fold.std()
print("CV MSE = mean of 5 folds = %.3f  (std %.3f)" % (cv_mse, cv_std))

## Visualize the data & results

_How good is the model when every slice gets a turn as the test set?_

### Step 1 — Cross-validate a classifier on real data

We switch to 569 real breast-cancer patients with 30 features each, classifying malignant vs benign. We standardise the features and use **StratifiedKFold**, which keeps each fold's class balance the same as the full dataset — important when one class is more common than the other. Each fold reports accuracy on patients the model never trained on.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

# REAL data: 569 breast-cancer patients, 30 features, malignant vs benign.
bc = load_breast_cancer()
X = StandardScaler().fit_transform(bc.data)

# Stratified folds keep each fold's class balance like the whole set.
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(
    LogisticRegression(max_iter=5000), X, bc.target,
    cv=skf,
    scoring='accuracy',
)

### Step 2 — Plot per-fold accuracy and the mean

A bar per fold shows how the accuracy varies depending on which slice was held out. We add one more bar for the **mean** — the headline cross-validation score — in a different colour. Bars that sit close together mean the model performs consistently no matter which patients it is tested on.

In [ ]:
labels = ['fold 1', 'fold 2', 'fold 3', 'fold 4', 'fold 5', 'MEAN']
vals = list(scores) + [scores.mean()]
colors = ['#4ea1ff'] * 5 + ['#7ee787']

plt.bar(labels, vals, color=colors)
plt.ylabel('accuracy')
plt.title('Breast-cancer logistic regression: per-fold accuracy and mean')
plt.show()